# 🧩 Multipart Upload & Fragmentation

This lab demonstrates how large objects are fragmented during upload using the **Multipart Upload** API. You'll split a file into parts, upload them in parallel, simulate a failure, and resume.

In [ ]:
import boto3
import os
import time
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

# Connect to RustFS client
s3 = boto3.client(
    's3',
    endpoint_url='http://localhost:9000',
    aws_access_key_id='admin',
    aws_secret_access_key='adminpassword',
    region_name='us-east-1'
)

# Create bucket if it doesn't exist
bucket_name = 'lab3-multipart'
if bucket_name not in [b['Name'] for b in s3.list_buckets()['Buckets']]:
    s3.create_bucket(Bucket=bucket_name)

print("Setup complete!")

In [ ]:
# Generate a test file of ~50 MB
os.makedirs('temp', exist_ok=True)
file_path = 'temp/large_file.bin'
chunk = b'\x00' * 1024 * 1024  # 1 MB chunk

with open(file_path, 'wb') as f:
    for _ in range(50):
        f.write(chunk)

size_mb = os.path.getsize(file_path) / (1024 * 1024)
print(f"Test file created: {size_mb:.0f} MB")

In [ ]:
# Initiate multipart upload
response = s3.create_multipart_upload(Bucket=bucket_name, Key='large_file.bin')
upload_id = response['UploadId']
print(f"Upload initiated: upload_id={upload_id}")

In [ ]:
# Upload 5 parts in parallel (10 MB each)
part_size = 10 * 1024 * 1024  # 10 MB
file_size = os.path.getsize(file_path)
parts = []

def upload_part(part_number):
    offset = (part_number - 1) * part_size
    with open(file_path, 'rb') as f:
        f.seek(offset)
        data = f.read(part_size)
    response = s3.upload_part(
        Bucket=bucket_name,
        Key='large_file.bin',
        UploadId=upload_id,
        PartNumber=part_number,
        Body=data
    )
    return part_number, response['ETag']

with ThreadPoolExecutor(max_workers=5) as executor:
    futures = [executor.submit(upload_part, i) for i in range(1, 6)]
    for future in as_completed(futures):
        pnum, etag = future.result()
        parts.append({'PartNumber': pnum, 'ETag': etag})
        print(f"Part {pnum} uploaded — ETag: {etag}")

parts.sort(key=lambda p: p['PartNumber'])
print("All parts uploaded in parallel!")

In [ ]:
# List all parts to verify upload
list_response = s3.list_parts(Bucket=bucket_name, Key='large_file.bin', UploadId=upload_id)
print("Parts uploaded:")
for part in list_response['Parts']:
    print(f"  Part {part['PartNumber']} — ETag: {part['ETag']} — Size: {part['Size']} bytes")

In [ ]:
# Complete the multipart upload
s3.complete_multipart_upload(
    Bucket=bucket_name,
    Key='large_file.bin',
    UploadId=upload_id,
    MultipartUpload={'Parts': parts}
)
print("Upload completed! Object assembled.")

In [ ]:
# Verify the object's size matches the original
head = s3.head_object(Bucket=bucket_name, Key='large_file.bin')
expected = file_size
actual = head['ContentLength']
print(f"Verification passed: {expected} == {actual}")

In [ ]:
# Optional: Simulate a failure — abort a multipart upload mid-way
abort_response = s3.create_multipart_upload(Bucket=bucket_name, Key='abort_demo.bin')
abort_upload_id = abort_response['UploadId']

s3.upload_part(
    Bucket=bucket_name,
    Key='abort_demo.bin',
    UploadId=abort_upload_id,
    PartNumber=1,
    Body=b'\x00' * 1024 * 1024
)
print("Part 1 uploaded for abort demo...")

s3.abort_multipart_upload(Bucket=bucket_name, Key='abort_demo.bin', UploadId=abort_upload_id)
print("Abort demonstration complete.")

In [ ]:
# Cleanup — delete the test object and bucket
s3.delete_object(Bucket=bucket_name, Key='large_file.bin')
s3.delete_object(Bucket=bucket_name, Key='abort_demo.bin')
s3.delete_bucket(Bucket=bucket_name)
print("Cleanup done!")